# 06 · Verificación / Crítica (Evaluación)
Evalúa automáticamente la respuesta generada según tres criterios:

| Criterio | Descripción | Umbral mínimo |
|---|---|---|
| **Grounding** | ¿Está respaldada por el contexto? | 0.7 |
| **Coherencia** | ¿Es lógica y libre de alucinaciones? | 0.7 |
| **Completitud** | ¿Responde completamente la pregunta? | 0.7 |

Si algún criterio no supera el umbral, el sistema solicita regeneración
(loop controlado, máximo 3 intentos).

### ¿Por qué Gemini aquí?
La verificación es la tarea más exigente: requiere **razonamiento
multi-criterio** —comparar la respuesta contra el contexto, detectar
alucinaciones sutiles y evaluar completitud semántica. Gemini 2.0 Flash
supera a Groq en este tipo de razonamiento crítico complejo.


In [ ]:
!pip install -q \
    langchain==0.3.14 \
    langchain-google-genai==2.0.8 \
    pydantic==2.10.4
print('✓ Dependencias instaladas')


In [ ]:
from google.colab import userdata

GEMINI_KEY   = userdata.get('GEMINI_API_KEY')
GEMINI_MODEL = 'gemini-2.0-flash'


In [ ]:
from pydantic import BaseModel, Field

class ResultadoVerificacion(BaseModel):
    """Esquema de salida estructurada del agente verificador."""
    aprobado: bool = Field(
        description='True si TODOS los criterios superan 0.7'
    )
    puntuacion_grounding: float = Field(
        ge=0.0, le=1.0,
        description='Fracción de afirmaciones respaldadas por el contexto'
    )
    puntuacion_coherencia: float = Field(
        ge=0.0, le=1.0,
        description='Nivel de coherencia lógica y ausencia de alucinaciones'
    )
    puntuacion_completitud: float = Field(
        ge=0.0, le=1.0,
        description='Grado en que se responde completamente la pregunta'
    )
    problemas: list[str] = Field(
        default=[],
        description='Lista de problemas específicos detectados'
    )
    sugerencias: str = Field(
        default='',
        description='Instrucciones para mejorar la respuesta si no aprobó'
    )


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

def crear_verificador(api_key: str):
    """
    Construye el agente verificador con salida estructurada Pydantic.

    Evalúa la respuesta RAG contra el contexto recuperado y la pregunta
    original. Usa temperatura=0 para reproducibilidad.

    Args:
        api_key : Google Gemini API key.

    Returns:
        Función `verificar(query, respuesta, contexto) -> ResultadoVerificacion`.
    """
    llm    = ChatGoogleGenerativeAI(model=GEMINI_MODEL,
                                    google_api_key=api_key,
                                    temperature=0.0)
    parser = PydanticOutputParser(pydantic_object=ResultadoVerificacion)

    prompt = ChatPromptTemplate.from_messages([
        ('system', """Eres un verificador crítico de respuestas RAG para un sistema biomédico.

CRITERIOS DE EVALUACIÓN (escala 0.0 – 1.0):
1. GROUNDING    : ¿Cada afirmación está respaldada por el contexto?
2. COHERENCIA   : ¿Es lógica? ¿Hay contradicciones o datos inventados?
3. COMPLETITUD  : ¿Responde completamente la pregunta original?

REGLA: aprobado = True solo si los tres criterios >= 0.7

{format_instructions}

CONTEXTO RECUPERADO:
{contexto}"""),
        ('human', 'PREGUNTA ORIGINAL:\n{query}\n\nRESPUESTA A EVALUAR:\n{respuesta}')
    ])

    chain = prompt | llm | parser

    def verificar(query: str, respuesta: str,
                  contexto: str) -> ResultadoVerificacion:
        """
        Evalúa la calidad de la respuesta generada.

        Args:
            query     : Pregunta original del usuario.
            respuesta : Respuesta generada por el módulo de generación.
            contexto  : Contexto de fragmentos usados para generar.

        Returns:
            ResultadoVerificacion con puntuaciones y veredicto.
        """
        return chain.invoke({
            'query'              : query,
            'respuesta'          : respuesta,
            'contexto'           : contexto[:3500],
            'format_instructions': parser.get_format_instructions()
        })

    return verificar


In [ ]:
# ── Demo con loop de regeneración controlado ──────────────────────────────
verificar = crear_verificador(GEMINI_KEY)

UMBRAL    = 0.7
MAX_LOOPS = 3

query_demo = '¿Cómo se usa el aprendizaje profundo para predecir efectos de fármacos?'
respuesta_demo = (
    'El deep learning ha revolucionado el descubrimiento de fármacos. '
    'Los modelos de grafos (GNN) se usan para predecir interacciones '
    'entre moléculas y proteínas [1][2]. También se aplican redes '
    'recurrentes para modelar secuencias biológicas [3].'
)
contexto_demo = (
    '[1] Deep learning for drug discovery... GNN models predict binding affinity. '
    '[2] Graph neural networks in computational biology... '
    '[3] LSTM models for genomic sequence analysis...'
)

for intento in range(1, MAX_LOOPS + 1):
    print(f'\n--- Verificación: intento {intento}/{MAX_LOOPS} ---')
    r = verificar(query_demo, respuesta_demo, contexto_demo)

    print(f'  Aprobado    : {r.aprobado}')
    print(f'  Grounding   : {r.puntuacion_grounding:.2f}')
    print(f'  Coherencia  : {r.puntuacion_coherencia:.2f}')
    print(f'  Completitud : {r.puntuacion_completitud:.2f}')
    if r.problemas:
        print(f'  Problemas   : {r.problemas}')
    if r.sugerencias:
        print(f'  Sugerencias : {r.sugerencias[:120]}')

    if r.aprobado:
        print('\n✓ RESPUESTA APROBADA')
        break
    if intento == MAX_LOOPS:
        print('\n⚠ Máximo de intentos alcanzado — se retorna la última versión')
